In [10]:
!pip install seleniumbase curl_cffi pandas

  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 8.3 MB/s eta 0:00:00
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 6.2 MB/s eta 0:00:00a 0:00:01m
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.4 MB/s eta 

In [15]:
import os
import time
import re
import requests
import pandas as pd
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- Configuration --------------------
CNEL_URL = "https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL"
BASE_DIR = "cnel_fresh_downloads"
PDF_DIR = os.path.join(BASE_DIR, "documents")
CSV_PATH = os.path.join(BASE_DIR, "download_log.csv")

# Set this to 1 if you want to start from the very beginning
# If you want to start at page 38, the script will click "Next" until it gets there.
TARGET_START_PAGE = 38 

DOWNLOAD_EXECUTOR = ThreadPoolExecutor(max_workers=5)

def make_driver():
    opts = Options()
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.page_load_strategy = 'normal'
    prefs = {"profile.managed_default_content_settings.images": 2}
    opts.add_experimental_option("prefs", prefs)
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()

def download_file_background(url, filepath, session):
    try:
        time.sleep(0.5)
        response = session.get(url, stream=True, timeout=30)
        if response.status_code == 200:
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=65536): 
                    f.write(chunk)
            print(f"      [✓] Saved: {os.path.basename(filepath)}")
    except: pass

def scrape_italy_all():
    os.makedirs(PDF_DIR, exist_ok=True)
    
    existing_urls = set()
    if os.path.exists(CSV_PATH):
        try:
            df_existing = pd.read_csv(CSV_PATH)
            if not df_existing.empty:
                url_col = [c for c in df_existing.columns if c.lower() == 'url'][0]
                existing_urls = set(df_existing[url_col].astype(str).tolist())
        except: pass

    if not os.path.exists(CSV_PATH):
        pd.DataFrame(columns=["Page", "Codice", "Titolo", "Tipologia", "URL", "Filename", "Status"]).to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

    d = make_driver()

    try:
        print(f"Opening {CNEL_URL}...")
        d.get(CNEL_URL)
        
        # --- 1. CLICK SEARCH ---
        print("Clicking 'Cerca'...")
        search_btn = WebDriverWait(d, 20).until(
            EC.element_to_be_clickable((By.ID, "dnn_ctr905_View_btnSearch"))
        )
        d.execute_script("arguments[0].click();", search_btn)
        
        # Wait for initial table
        WebDriverWait(d, 30).until(EC.presence_of_element_located((By.XPATH, "//table//tr[td]")))

        # Create Session for downloads
        session = requests.Session()
        for cookie in d.get_cookies():
            session.cookies.set(cookie['name'], cookie['value'])
        session.headers.update({'User-Agent': d.execute_script("return navigator.userAgent;"), 'Referer': CNEL_URL})

        current_page = 1
        
        # --- 2. PAGE BY PAGE LOOP ---
        while current_page <= 500:
            # Check if we are at or past the page we actually want to scrape
            if current_page >= TARGET_START_PAGE:
                print(f"\n--- Scraping Page {current_page} ---")
                
                # Wait for links to be stable
                WebDriverWait(d, 30).until(EC.presence_of_element_located((By.XPATH, "//a[contains(@href, 'attachment')]")))
                download_links = d.find_elements(By.XPATH, "//a[contains(@href, 'attachment')]")
                    
                page_data = []
                for link in download_links:
                    try:
                        url = link.get_attribute("href")
                        if url in existing_urls: continue
                        
                        row = link.find_element(By.XPATH, "./ancestor::tr")
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) < 5: continue
                        
                        titolo, tipologia, codice = cols[1].text.strip(), cols[2].text.strip(), cols[3].text.strip()
                        ext = ".pdf"
                        if ".p7m" in url.lower(): ext = ".pdf.p7m"
                        elif ".rtf" in url.lower(): ext = ".rtf"
                        
                        filename = sanitize_filename(f"{codice} - {titolo[:55]}{ext}")
                        filepath = os.path.join(PDF_DIR, filename)
                        
                        if not os.path.exists(filepath):
                            DOWNLOAD_EXECUTOR.submit(download_file_background, url, filepath, session)
                        
                        page_data.append([current_page, codice, titolo, tipologia, url, filename, "Downloaded"])
                        existing_urls.add(url)
                    except: continue

                if page_data:
                    pd.DataFrame(page_data).to_csv(CSV_PATH, mode='a', header=False, index=False, encoding="utf-8-sig")
            else:
                print(f"Skipping Page {current_page} (Target is {TARGET_START_PAGE})...")

            # --- 3. GO TO NEXT PAGE ---
            try:
                # Find the link to the right of the current <span>
                next_btn_xpath = "//tr[contains(@class, 'pagination')]//span/parent::td/following-sibling::td/a"
                next_btns = d.find_elements(By.XPATH, next_btn_xpath)
                
                if next_btns:
                    target_btn = next_btns[0]
                    old_page_num = current_page
                    
                    d.execute_script("arguments[0].scrollIntoView({block: 'center'});", target_btn)
                    time.sleep(1)
                    d.execute_script("arguments[0].click();", target_btn)
                    
                    # WAIT specifically for the span text to change to the next number
                    # This prevents the script from scraping the same page twice
                    WebDriverWait(d, 30).until(
                        lambda driver: driver.find_element(By.XPATH, "//tr[contains(@class, 'pagination')]//span").text != str(old_page_num)
                    )
                    
                    # Update current_page variable
                    new_span = d.find_element(By.XPATH, "//tr[contains(@class, 'pagination')]//span")
                    current_page = int(new_span.text)
                else:
                    print("Reached end of archive.")
                    break
            except Exception as e:
                print(f"Navigation stopped: {e}")
                break

    finally:
        d.quit()
        DOWNLOAD_EXECUTOR.shutdown(wait=True)
        print("Done.")

if __name__ == "__main__":
    scrape_italy_all()

Opening https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL...
Clicking 'Cerca'...
Done.


AttributeError: 'NoneType' object has no attribute 'is_displayed'